In [ ]:
from db import ImageDatabase
import pandas as pd
import numpy as np
from PIL import Image
import sys
from pathlib import Path
from tqdm.notebook import tqdm
import random
import time

tqdm.pandas()

src_path ='Text_to_image_search'
sys.path.insert(0, str(src_path))

from meme_embed import load_embedder, from_pretrained


DS_PATH = Path('dataset')
DS_IM_PATH = DS_PATH / 'images'

# Load 

In [ ]:
emb = load_embedder("clip-jina-v2", embedding_dim=512)   # or just load_embedder()

odf = pd.read_json(DS_PATH / 'metadata.jsonl', lines=True)

db = ImageDatabase("db.db")

In [ ]:
if 0: # Reload from db
    ls = []
    for i in tqdm(db.get_ids_by_fields(img_emb=True, ocr_emb=True, ocr_text=True)):
        r = db.get_record(i)
        d ={
            'id': i,
            'image_embedding': r['img_emb'],
            'text_embedding': r['ocr_emb'],
            'ocr_text': r['ocr_text'],
            'file_name': f'{i}.webp'
        }
        ls.append(d)
    odf = pd.DataFrame(ls)
    odf

# Process emb

In [ ]:
processed_im_ids = db.get_ids_by_fields(img_emb=True)
processed_im_ids

In [ ]:
if 0:
    print('Skipping:', len(processed_im_ids))
    for img_id in (pbar:=tqdm(odf[~odf['id'].isin(processed_im_ids)]['id'].tolist())):
        img_pth = DS_IM_PATH / f'{img_id}.webp'

        img_emb = emb.encode_image(img_pth)
        db.upsert(img_id, img_emb=img_emb)

    print('Done')

In [ ]:
BATCH_SIZE = 16
MAX_DIM = 700**2

print('Skipping:', len(processed_im_ids))

queue = []

def img_preprocess(img: Image.Image) -> Image.Image:
    sz = img.size
    MAX_DIM = 600**2
    d = int(np.ceil(np.log2(max(sz[0]*sz[1] / MAX_DIM, 1))))
    if d:
        f = 2**d
        img = img.resize((sz[0]//f, sz[1]//f))
    return img

def process_queue():
    global queue

    if len(queue) == 0:
        return

    imgs = [img_preprocess(Image.open(DS_IM_PATH / f'{i}.webp')) for i in queue]

    emb_batch = emb.encode_image(imgs)

    for id_, embed in zip(queue, emb_batch):
        db.upsert(id_, img_emb=embed)

    queue = []

for i in (pbar:=tqdm(odf[~odf['id'].isin(processed_im_ids)]['id'].tolist())):
    queue.append(i)

    if len(queue) >= BATCH_SIZE:
        start = time.perf_counter()
        process_queue()
        end = time.perf_counter()
        pbar.set_description(f'Speed: {(end - start) / BATCH_SIZE: .2f}s/im')
process_queue()

# Text processing

## OCR TEST

In [ ]:
import torch
import pandas as pd
from tqdm.notebook import tqdm  # Use the notebook version
import numpy as np
from db import ImageDatabase
from PIL import Image
import sys
from pathlib import Path
import random
import time

tqdm.pandas()

src_path ='Text_to_image_search'
sys.path.insert(0, str(src_path))

from meme_embed import load_embedder, from_pretrained
import ocr


DS_PATH = Path('dataset/')
DS_IM_PATH = DS_PATH / 'images'

In [ ]:
# rec = ocr.recognize(img)

def get_text(r: pd.DataFrame) -> str:
    im_id = r['id']
    im_pt = DS_IM_PATH / f'{im_id}.webp'

    return ocr.read_text(Image.open(im_pt))

odf['extracted_ocr'] = odf.progress_apply(get_text, axis=1)

In [ ]:
df = pd.read_json(DS_PATH / 'metadata.jsonl', lines=True)
df

In [ ]:
odf['ocr_text'] = np.where(odf['extracted_ocr']=='', df['image_desc'] + '. ' + df['meaning'], odf['extracted_ocr'])
odf

In [ ]:
# add to db 
for _, r in tqdm(odf[['id', 'ocr_text']].iterrows()):
    iid, itxt = r['id'], r['ocr_text']

    db.upsert(iid, ocr_text=itxt)

## Embeddings

In [ ]:

processed_im_ids = db.get_ids_by_fields(ocr_emb=True)

print('Skipping:', len(processed_im_ids))

for _, i in (pbar:=tqdm(odf[~odf['id'].isin(processed_im_ids)][['id', 'ocr_text']].iterrows())):
    id_, ocr_text = i['id'], i['ocr_text']
    
    ocr_emb = emb.encode_text(ocr_text)

    db.upsert(id_, ocr_text=ocr_text, ocr_emb=ocr_emb)

# EXport

In [ ]:
ls = []
for i in tqdm(db.get_ids_by_fields(img_emb=True, ocr_emb=True, ocr_text=True)):
    r = db.get_record(i)
    d ={
        'id': i,
        'image_embedding': r['img_emb'],
        'text_embedding': r['ocr_emb'],
        'ocr_text': r['ocr_text'],
        'file_name': f'{i}.webp'
    }
    ls.append(d)
df = pd.DataFrame(ls)
df

## Quantization

In [ ]:
MAT_SHRINK = 256
def quantize_to_int8(x: np.ndarray) -> tuple[np.ndarray, float]:
    absmax = np.abs(x).max()
    scale = absmax / 127.0 if absmax != 0 else 1.0
    quantized = np.clip(np.round(x / scale), -128, 127).astype(np.int8)[:MAT_SHRINK]
    
    return quantized

def dequantize_from_int8(quantized: np.ndarray, scale: float) -> np.ndarray:
    return quantized.astype(np.float32) * scale

img_emb_max = np.abs(np.stack(df['image_embedding'].to_numpy())).max()
txt_emb_max = np.abs(np.stack(df['text_embedding'].to_numpy())).max()

print(img_emb_max, txt_emb_max)

df['image_embedding_q'] = (df['image_embedding']).progress_apply(quantize_to_int8)
df['text_embedding_q'] = (df['text_embedding']).progress_apply(quantize_to_int8)
df

## Save

In [ ]:
edf = df[['id', 'image_embedding_q', 'text_embedding_q', 'ocr_text', 'file_name', 'image_embedding', 'text_embedding']].rename(columns={
        'image_embedding_q': 'image_embedding', 
        'text_embedding_q': 'text_embedding', 
        'image_embedding': 'image_embedding', 
        'text_embedding': 'text_embedding_raw',
        'image_embedding': 'image_embedding_raw'
    })

edf.to_parquet('sample.parquet')
edf